# Finding Symbols


Finding the ticker symbol, security identifier, the sector, and other metadata is easy if you know where to look.  This guide is intended to introduce some methods for searching, screening, and discovery.

For maximum coverage and functionality, install OpenBB with `[all]` packages.

The examples here will assume that the OpenBB Platform has been installed, the environment is active, and it has been imported into a Python session.  If the installation is fresh, or an extension was just installed, the Python interface will need to be rebuilt.  It will only take a few moments to complete.

In [ ]:
from openbb import obb

The simplest way to find tickers is with a basic text query.

## Search Nasdaq

In [ ]:
obb.equity.search("JPMorgan", provider="nasdaq").to_df().head(3)

## Search Cboe

In [ ]:
obb.index.search("SPX", provider="cboe").to_df().tail(5)

## Search ETFs

In [ ]:
obb.etf.search("gold", provider="tmx").to_df().iloc[-5:]

In [ ]:
obb.etf.search("covered call", provider="fmp").to_df().iloc[:5]

## Search the SEC

Use an empty string, `""`, to return the complete list - over 10,000.

The SEC sorts this list by market cap.  Applying the `to_df()` method to `all_companies` will show them from biggest-to-smallest.

In [ ]:
all_companies = obb.equity.search("", provider="sec")

print(len(all_companies.results))

all_companies.to_df().head(10)

### Find an Institution

Some reporting companies, like invesment trusts and insurance companies, do not have a ticker symbol directly associated with them.  Filers in the US will have a CIK number, used to retrieve documents from the SEC.

In [ ]:
instututions = obb.regulators.sec.institutions_search("Berkshire Hathaway").to_df()
instututions

### Find a Filing

Search for filings by CIK or ticker symbol.

In [ ]:
homestate_filings = obb.equity.fundamental.filings(cik="0000829771", provider="sec")

homestate_filings.to_df().iloc[-1]

Or, search by form type.

In [ ]:
aapl_filings = obb.equity.fundamental.filings("AAPL", type="4", provider="sec")

aapl_filings.to_df().iloc[0]

## Screen Markets

Screeners provide a targeted search, a tool for comparison and discovery.  Find stocks from around the world with the screener endpoint, and the `openbb-fmp` provider.

### Find Stocks From India

In [ ]:
results = obb.equity.screener(country="IN", provider="fmp").to_df()
display(len(results))
results.head(5).convert_dtypes()

In [ ]:
# The Nasdaq screener is limited to the American market listings.
results = obb.equity.screener(country="india", provider="nasdaq").to_df()
display(len(results))
results.head(5).convert_dtypes()

### Search by Sector

In [ ]:
sector_results = obb.equity.screener(
    country="IN", sector="financial_services", provider="fmp"
).to_df()
display(len(sector_results))
sector_results.head(5).convert_dtypes()

In [ ]:
# The same can be done with the Nasdaq provider, covering the American market.
sector_results = obb.equity.screener(
    sector="financial_services", provider="nasdaq"
).to_df()
display(len(sector_results))
sector_results.head(5).convert_dtypes()

### Search by Industry

In [ ]:
industry_results = obb.equity.screener(
    industry="apparel_manufacturing", provider="finviz"
).to_df()
display(len(industry_results))
industry_results.head(5)

In [ ]:
industry_results = obb.equity.screener(
    industry="manufacturing", provider="fmp", country="IN"
).to_df()
display(len(industry_results))
industry_results.head(5)

### Search by Exchange

Some countries, like America, have multiple exchanges.  Narrow the search by combining two or more parameters.  The example below finds the companies listed on the American Stock Exchange (AMEX) that are domiciled in China.

In [ ]:
exchange_results = obb.equity.screener(
    exchange="amex", country="CN", provider="fmp"
).to_df()
display(len(exchange_results))
exchange_results

### Filter ADRs

Use the Nasdaq screener to get only American Depositary Receipts

In [ ]:
obb.equity.screener(exsubcategory="adr", provider="nasdaq").to_df()

### Filter by Metric

Applying some filters refines and targets the search. The example below finds listing on the NYSE domiciled in the USA, with a market cap between $100-300 billion, and exhibiting a beta value of less than 0.5

In [ ]:
obb.equity.screener(
    exchange="nyse",
    mktcap_min=100000000000,
    mktcap_max=300000000000,
    country="us",
    beta_max=0.5,
    provider="fmp",
).to_df()

### Finviz Screener

The `openbb-finviz` provider extension supports screener presets from V3 SDK and Terminal. See the details here: [https://pypi.org/project/openbb-finviz/](https://pypi.org/project/openbb-finviz/)

In [ ]:
obb.equity.screener(
    metric="overview", signal="top_gainers", provider="finviz", mktcap="mid_over"
).to_df()

## Get Available Indices

List all indices from a source with:

In [ ]:
indices = obb.index.available(provider="yfinance").to_df()
print(len(indices))

indices[indices["name"].str.contains("ASX 200")]

Filter the list down by querying the DataFrame.

With the `openbb-yfinance` extension, index time series can be  loaded using the ticker symbol or short code.  Non-American indices have a code beginning with the two-letter country code.

In [ ]:
(
    obb.index.price.historical("au_utilities", provider="yfinance").to_df().tail(1)
    == obb.index.price.historical("^AXUJ", provider="yfinance").to_df().tail(1)
)